In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from main import load_cifar_batch

In [ ]:
def show_images(X, y=None, class_names=None, num_images=9):
    """
    Displays random images in a grid.
    
    Parameters:
    - X: numpy array of shape (N, H, W, C)
    - y: labels (optional)
    - class_names: list of class names (optional)
    - num_images: how many images to show
    """
    
    N = X.shape[0]
    indices = np.random.choice(N, num_images, replace=False)
    
    # if num_images%2 == 0 else grid_size  
    grid_size = int(np.sqrt(num_images)) 
    
    plt.figure(figsize=(8, 8))
    
    for i, idx in enumerate(indices):
        plt.subplot(grid_size, grid_size, i + 1)
        plt.imshow(X[idx].astype(np.uint8))
        plt.axis("off")
        
        if y is not None:
            if class_names is not None:
                plt.title(class_names[y[idx]])
            else:
                plt.title(str(y[idx]))
    
    plt.tight_layout()
    plt.show()
    

In [ ]:
def batch_sample(x, sample_batch):
    batch_indices = np.random.choice(x.shape[0], int(sample_batch), replace=False)
    return batch_indices

In [ ]:
np.random.seed(42)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
X, y = load_cifar_batch("datasets/cifar-10-batches-py/data_batch_1")
X_normalized = X/255.0
X_flat = X_normalized.reshape(X_normalized.shape[0], -1)
x_valid = X_normalized[:500]
y_valid = y[:500]

# Better weight initialization (smaller, from normal distribution)
W = np.random.randn(len(class_names), X_flat.shape[1]) * 0.01
b = np.zeros(shape=(1, 10))


# Convert labels to one-hot encoding
Y = np.eye(10)[y]  # Shape: (N, 10)

# print(y.shape)

# print(Y[0].argmax())

# Y[0].shape

l = np.random.randint(1,10,size=20)  # Random scores for 10 classes
l
e = np.eye(10)[l] 
e
# print(l)
# l = np.eye(10)[np.argmax(l, axis=1)]
# l


In [ ]:
'''
normalized image pixels for stabillity result   
'''
def model(x,W,b):
    raw_prob = (np.dot(x,W)) + b
    return raw_prob

In [ ]:

'''
This can cause all outputs to be 0 or NaN if the values in z are large (positive or negative), because np.
exp(z) can overflow (resulting in inf or NaN) or underflow (resulting in 0). When you divide by a 
sum of inf or 0, you get NaN or 0.

solution -> numerically stable 
'''
def softmax(z):
    z -= np.max(z)
    return np.exp(z) / np.sum(np.exp(z))

In [ ]:
def compute_cost(W, X_train, b, y):
    """
    Computes the cross-entropy cost without any for loops.
    y can be either integer labels or one-hot encoded
    """
    N = X_train.shape[0]
    
    # 1. Forward pass for ALL images simultaneously
    # print(f'X_train shape: {X_train.shape}, W shape: {W.shape}, b shape: {b.shape}')
    scores = np.dot(X_train, W.T) + b
    
    # Vectorized Softmax with stability 
    scores -= np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # Handle both one-hot encoded and integer labels
    if y.ndim == 2:  # One-hot encoded
        correct_class_probs = np.sum(probs * y, axis=1)
    else:  # Integer labels
        correct_class_probs = probs[np.arange(N), y]
    
    # 4. Calculate the loss for all N images at once
    losses = -np.log(correct_class_probs + 1e-15)
    
    # 5. Calculate the final average cost
    total_loss = np.sum(losses) / N
    
    # If you still want the history of individual losses, 'losses' already has them!
    hist = losses.tolist() 
    
    return total_loss, hist

total_loss,hist = compute_cost(W, X_flat, b, y)



# Convert the Python list back to a NumPy array so we can sort it easily
hist_array = np.array(hist)
sorted_hist = np.sort(hist_array)

# Create the X-axis (from 1 to N)
# Using len(sorted_hist) ensures it matches exactly, whether N is 100 or 10000
x_axis = np.arange(1, len(sorted_hist) + 1)

# 4. Plot the distribution
plt.plot(x_axis, sorted_hist)
plt.title("Distribution of Image Losses in the Batch")
plt.xlabel("Images (Sorted from Easiest to Hardest)")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True)
plt.show()

print("Top 10 highest losses:", sorted_hist[-10:])

In [ ]:
#evalute grad
def compute_grad(X, W, b, 
                 y_true, reg=0.01
):
    """
    Computes the analytical gradient.
    Assumes X is shape (N, D), W is (C, D), y_true is (N, C) one-hot encoded.
    """
    N = X.shape[0] 
    
    # Forward pass to get predictions (z)
    scores = np.dot(X, W.T) + b
    
    # Softmax per-sample (row-wise) with stability
    scores -= np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    z = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # Calculate the error (derivative of loss w.r.t scores)
    dz = z - y_true 
    
    # THE CHAIN RULE: Multiply error by inputs (Vectorized)
    # This replaces the entire for-loop
    dl_dw = np.dot(dz.T, X)
    dl_db = np.sum(dz, axis=0, keepdims=True)
    
    #Average over batch (divide by N)
    dl_dw /= N
    dl_db /= N
    
    #Add the Regularization Gradient (reg * W, not reg/2)
    dl_dw += reg * W

    return dl_dw, dl_db

In [ ]:
np.random.seed(42)
def SGD_minibatch(X, W, b, y, 
                  compute_cost=compute_cost, 
                  compute_grad=compute_grad,
                  learning_rate=1e-5, 
                  batch_size=256,
                  reg=0.001,
                  epochs=10):
    
    loss_history = []
    N = X.shape[0]
    
    for epoch in range(epochs):
        # Shuffle data for each epoch
        indices = np.random.permutation(N)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        # Iterate through mini-batches
        for i in range(0, N, int(batch_size)):
            # Get a batch
            X_batch = X_shuffled[i:i+int(batch_size)]
            y_batch = y_shuffled[i:i+int(batch_size)]
            
            # Compute gradients and loss for this batch
            dw, db = compute_grad(X_batch, W, b, y_batch, reg)
            loss, _ = compute_cost(W, X_batch, b, y_batch)
            loss_history.append(loss)
            
            # Update weights
            W -= learning_rate * dw
            b -= learning_rate * db
        
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss:.4f}')
    
    return W, b, loss_history

# Usage:
new_w, new_b, hist = SGD_minibatch(X_flat, W, b, Y, 
                                    batch_size=256, 
                                    epochs=10)

plt.subplot(1, 2, 1)
plt.plot(hist[::-1])

In [ ]:
X_test,y_test = load_cifar_batch("datasets/cifar-10-batches-py/test_batch")
#Take some samples from the test set and show them
samples = 10
for i in range(samples):
    plt.subplot(2,5,i+1)
    plt.imshow((X_flat[i]*255).astype(np.uint8))
    plt.axis("off")
    plt.title(class_names[y[i]])
plt.tight_layout()
plt.show() 




In [ ]:
np.random.seed(42)
M = np.arange(0,20)
prob = np.random.uniform(0,255,size= (1,20)) 
prob/=255.
s = softmax(prob)


In [ ]:
#.transpose(0, 2, 3, 1)
X_img = X_flat.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
X_img.shape

In [ ]:
# Tiny index-mapping demo: flat (3072,) -> (3,32,32) -> (32,32,3)
x_flat_demo = np.arange(3072)
print("flat shape:", x_flat_demo.shape)

# Step 1: split flat vector into channel-first image
x_chw = x_flat_demo.reshape(3, 32, 32)
print("after reshape (C,H,W):", x_chw.shape)

# Step 2: move channel axis to the end for plotting
x_hwc = x_chw.transpose(1, 2, 0)
print("after transpose (H,W,C):", x_hwc.shape)

# Check exact positions to see how values move
print("R(0,0): flat[0]    -> x_chw[0,0,0] -> x_hwc[0,0,0] =", x_flat_demo[0], x_chw[0,0,0], x_hwc[0,0,0])
print("G(0,0): flat[1024] -> x_chw[1,0,0] -> x_hwc[0,0,1] =", x_flat_demo[1024], x_chw[1,0,0], x_hwc[0,0,1])
print("B(0,0): flat[2048] -> x_chw[2,0,0] -> x_hwc[0,0,2] =", x_flat_demo[2048], x_chw[2,0,0], x_hwc[0,0,2])

# Batch version (N,3072) -> (N,32,32,3)
X_demo = np.arange(2 * 3072).reshape(2, 3072)
X_img_demo = X_demo.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
print("batch shape:", X_demo.shape, "->", X_img_demo.shape)